# Lab 4 + Lab 5 Projesi: Kitap Tanıtım Yazısından Yazar Tahmini
Sabahattin Ali (0) ve Stefan Zweig (1) tahmini. **Bölüm 1:** lab4 (elle yazılmış Markov modeli). **Bölüm 2:** lab5 (sklearn Naive Bayes). Her bölüm kendi örnek notebook'unun hücre sırası ve fonksiyonlarıyla aynıdır.

Veri farklı olduğu için gerekli farklar `# FARK:` ile veya açıklama hücresinde belirtildi.

---
# Bölüm 1: lab4 (Markov modeli)

In [ ]:
import numpy as np
import pandas as pd
import string
import re
from collections import Counter

In [ ]:
df = pd.read_csv("Turkish_Book_Dataset_Kaggle_V2.csv", dtype=str)

In [ ]:
df.head()

In [ ]:
df.book_author.value_counts()[:10]

In [ ]:
yazar_0 = "Sabahattin Ali"
yazar_1 = "Stefan Zweig"

In [ ]:
# FARK: dosyada boş satırlar, çok kısa yazılar ve tekrarlar var, eliyoruz
df = df.dropna(subset=["book_detail", "book_author"])
df = df[df.book_detail.str.len() >= 50]
df = df.drop_duplicates("book_detail")

In [ ]:
kitaplar_0 = df[df["book_author"] == yazar_0]
kitaplar_1 = df[df.book_author == yazar_1]

In [ ]:
print(len(kitaplar_0), len(kitaplar_1))

In [ ]:
kitaplar_0.book_author.unique()

In [ ]:
kitaplar_1.book_author.unique()

In [ ]:
kitaplar_0 = kitaplar_0.book_detail.to_numpy()
kitaplar_1 = kitaplar_1.book_detail.to_numpy()

In [ ]:
print(yazar_0, "ilk kitabi:\n\n")
print(kitaplar_0[0])

In [ ]:
print(yazar_1, "ilk kitabi:\n\n")
print(kitaplar_1[0])

In [ ]:
kitaplar = [kitaplar_0, kitaplar_1]
yazarlar = [yazar_0, yazar_1]

In [ ]:
satirlar = []
labels = []

In [ ]:
# FARK: tanitim yazilarinda satir yok, bu yuzden cumlelere boluyoruz
# FARK: yazarin adini metinden siliyoruz (yoksa model adi okuyarak hile yapar)
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def kucult(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()

def temizle(satir, yazar):
    satir = kucult(satir).translate(tablo)
    isim_parcalari = [p for p in kucult(yazar).translate(tablo).split() if len(p) >= 3]
    return " ".join(k for k in satir.split() if not any(k.startswith(p) for p in isim_parcalari))

# kitap_seti: bir yazara ait tum kitaplar
for i, kitap_seti in enumerate(kitaplar):
    for kitap in kitap_seti:  # kitap: bir yazara ait kitap setindeki her bir kitabin tanitim yazisi
        for satir in re.split(r"(?<=[.!?])\s+|\n", kitap):
            # satiri "Hey! Naber?"den "hey naber"e donusturuyoruz
            satir = temizle(satir, yazarlar[i])
            if satir:
                satirlar.append(satir)
                labels.append(i)

In [ ]:
satirlar[:5]

In [ ]:
labels[:5]

In [ ]:
satirlar[-5:]

In [ ]:
labels[-5:]

In [ ]:
print(len(satirlar), len(labels))

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    satirlar, labels, test_size=0.2, random_state=42)

In [ ]:
print(len(X_train), len(y_train), len(X_test), len(y_test))

In [ ]:
for i in range(15):
    print(X_train[i], "::", y_train[i], "::", yazarlar[y_train[i]])

In [ ]:
for i in range(15):
    print(X_test[i], "::", y_test[i], "::", yazarlar[y_test[i]])

In [ ]:
w2i = {}  # kelime dağarcığı (vocab.) oluşturuyoruz
w2i["<UNK>"] = 0

In [ ]:
# FARK: geçiş tablosu kelime x kelime büyüklüğünde, bellek için sadece en sık geçen 3000 kelimeyi alıyoruz
sayac = Counter(token for satir in X_train for token in satir.split())

index = 1
for token, adet in sayac.most_common(3000):
    w2i[token] = index
    index += 1

In [ ]:
list(w2i.items())[:5]

In [ ]:
len(w2i)

In [ ]:
print(w2i["bir"], w2i["ve"], w2i["bu"])

In [ ]:
# "bir satranç ustası" satirini [14, 15, 16] listesine dönüştüreceğiz
X_train_int = []
X_test_int = []

In [ ]:
X_train[:3]

In [ ]:
# train setindeki text satirlari, int satirlara donusturuyoruz
for satir in X_train:
    satir_int = []
    tokens = satir.split()
    for token in tokens:
        # FARK: sozluk 3000 kelimeyle sinirli oldugu icin, train'de de sozlukte olmayan token'a <UNK> (0) veriyoruz
        satir_int.append(w2i.get(token, 0))

    X_train_int.append(satir_int)

In [ ]:
X_train_int[:3]

In [ ]:
# test setindeki satirlari, int satirlara donusturuyoruz
for satir in X_test:
    satir_int = []
    tokens = satir.split()
    for token in tokens:
        # eğer test setindeki bir token, train setinin sozlugunde w2i bulunmuyorsa
        # bu token için <UNK> tag'ine karsilik gelen 0 degerini atiyoruz
        satir_int.append(w2i.get(token, 0))

    X_test_int.append(satir_int)

In [ ]:
X_test_int[:3]

In [ ]:
X_train_int_SA = []  # sadece Sabahattin Ali'ye ait satirlar
for i, label in enumerate(y_train):
    if label == 0:  # 0 == Sabahattin Ali
        X_train_int_SA.append(X_train_int[i])

In [ ]:
len(X_train_int_SA)

In [ ]:
y_train_0s = [l for l in y_train if l == 0]

In [ ]:
len(y_train_0s)

In [ ]:
X_train_int_SA[:3]

In [ ]:
X_train_int_SZ = []  # Sadece Stefan Zweig'a ait satirlar
for i, label in enumerate(y_train):
    if label == 1:  # 1 == Stefan Zweig
        X_train_int_SZ.append(X_train_int[i])

print(len(X_train_int_SZ))

In [ ]:
M = len(w2i)  # dagarcik boyutu (vocab. size)
print(M)

# Sabahattin Ali modeli
pi_0 = np.ones(M)
A_0 = np.ones((M, M))

# Stefan Zweig modeli
pi_1 = np.ones(M)
A_1 = np.ones((M, M))

In [ ]:
def train_markov_model(satirlar_int, pi, A):
    for satir_int in satirlar_int:  # satir_int: [5928, 4336, 1535, 4397, 802]
        for i, token_index in enumerate(satir_int):  # i: 0, index: 5928
            if i == 0:
                # satir basindaysak, token'i pi'ye ekle
                pi[token_index] += 1
            else:
                # satir basinda degilsek,
                # bir onceki token'dan simdiki token'a gecis frekansini 1 arttir
                A[satir_int[i - 1], token_index] += 1

In [ ]:
train_markov_model(X_train_int_SA, pi_0, A_0)

In [ ]:
train_markov_model(X_train_int_SZ, pi_1, A_1)

In [ ]:
pi_0[:5]

In [ ]:
A_1[:5, :15]

In [ ]:
# normalizasyon
pi_0 = pi_0 / pi_0.sum()
pi_1 = pi_1 / pi_1.sum()

In [ ]:
pi_1[:5]

In [ ]:
test = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

test2 = test.sum(axis=0)

test3 = test.sum(axis=1)

test4 = test.sum(axis=1, keepdims=True)

print("original")
print(test)

print("-" * 20)
print("sum(axis=0)")
print(test2)

print("-" * 20)
print("sum(axis=1)")
print(test3)

print("-" * 20)
print("sum(axis=1, keepdims=True)")
print(test4)

In [ ]:
A_0 = A_0 / A_0.sum(axis=1, keepdims=True)
A_1 = A_1 / A_1.sum(axis=1, keepdims=True)

In [ ]:
log_pi_0 = np.log(pi_0)
log_A_0 = np.log(A_0)

log_pi_1 = np.log(pi_1)
log_A_1 = np.log(A_1)

In [ ]:
log_pi_1[:5]

In [ ]:
log_A_1[:5, :4]

In [ ]:
count_0 = sum(y == 0 for y in y_train)  # SA'ya ait satirlarin sayisi
count_1 = sum(y == 1 for y in y_train)  # SZ'ya ait satirlarin sayisi

total = len(y_train)  # train setindeki toplam satir sayisi

# prior'lari hesapla
p_0 = count_0 / total
log_p_0 = np.log(p_0)

p_1 = count_1 / total
log_p_1 = np.log(p_1)

In [ ]:
print(p_0, p_1, log_p_0, log_p_1)

In [ ]:
def compute_log_prob(input, clas):
    """ 
    input'un verilen class'a ait olma olasiliginin log degerini hesaplar

    örn input: [1, 3, 2, 7, ...] 
    örn clas: 0 veya 1
    """
    pi = log_pi_0
    A = log_A_0
    prior = log_p_0

    if clas == 1:
        pi = log_pi_1
        A = log_A_1
        prior = log_p_1

    log_prob = 0
    for i, word_index in enumerate(input):
        if i == 0:
            log_prob += pi[word_index]
        else:
            log_prob += A[input[i - 1], word_index]

    log_prob += prior
    return log_prob

In [ ]:
print(X_train_int[0], y_train[0])

In [ ]:
compute_log_prob(X_train_int[0], 0)

In [ ]:
compute_log_prob(X_train_int[0], 1)

In [ ]:
def predict(inputs):
    predictions = []
    for input in inputs:
        probas = [compute_log_prob(input, 0), compute_log_prob(input, 1)]
        prediction = np.argmax(probas)
        predictions.append(prediction)

    return predictions

In [ ]:
predictions_train = predict(X_train_int)
predictions_test = predict(X_test_int)

In [ ]:
predictions_train[:5]

In [ ]:
y_train[:5]

In [ ]:
predictions_train = np.array(predictions_train)
predictions_test = np.array(predictions_test)

In [ ]:
predictions_train == y_train

In [ ]:
train_accu = np.mean(predictions_train == y_train)
test_accu = np.mean(predictions_test == y_test)

In [ ]:
print("Train accuracy:", train_accu)
print("Test accuracy:", test_accu)

---
# Bölüm 2: lab5 (sklearn ile Naive Bayes)
Veri okuma ve temizleme Bölüm 1'de yapıldı, aynı `df` ile devam ediyoruz.

In [ ]:
import numpy as np 
import pandas as pd 
import string
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB

In [ ]:
df.book_author.value_counts()

In [ ]:
df_subset = df[df.book_author.isin(yazarlar)].copy()

In [ ]:
df_subset.book_author.value_counts()

In [ ]:
df_subset.head()

`clean` fonksiyonu şarkı notebook'undakiyle aynı mantıkta. Farklar: Türkçe `I/İ` harflerini doğru küçültür, tipografik tırnakları siler ve **yazarın adını metinden çıkarır** (tanıtım yazısında "Zweig", "Sabahattin Ali" geçiyor; çıkarmazsak model yazarın adını okuyarak hile yapar).

In [ ]:
tablo = str.maketrans("", "", string.punctuation + "…“”‘’")

def kucult(metin):
    return metin.replace("I", "ı").replace("İ", "i").lower()

def clean(row):
    satirlar = row["book_detail"].split("\n")
    metin = kucult(" ".join(satirlar)).translate(tablo)
    # yazarın adının parçalarını (ve zweigın, alinin gibi eklerini) sil
    isim_parcalari = [p for p in kucult(row["book_author"]).translate(tablo).split() if len(p) >= 3]
    kelimeler = [k for k in metin.split() if not any(k.startswith(p) for p in isim_parcalari)]
    return " ".join(kelimeler)

df_subset["clean"] = df_subset.apply(lambda row: clean(row), axis = 1)

In [ ]:
def etiketle(row):
    if row["book_author"] == yazarlar[0]:
        return 0 
    return 1

df_subset["labels"] = df_subset.apply(lambda row:  etiketle(row), axis=1)

In [ ]:
df_subset.head()

In [ ]:
df_subset.tail()

In [ ]:
X = df_subset.clean.to_numpy()
y = df_subset.labels.to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(len(X_train), len(X_test))

In [ ]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
len(vectorizer.vocabulary_)

In [ ]:
list(vectorizer.vocabulary_.items())[:10]

In [ ]:
vectorizer.get_feature_names_out()[:15]

In [ ]:
X_train

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

In [ ]:
model.score(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
predictions_train = model.predict(X_train)
print("Train F1:", f1_score(y_train, predictions_train))

predictions_test = model.predict(X_test)
print("Test F1:", f1_score(y_test, predictions_test))

In [ ]:
y_train[:15]

In [ ]:
predictions_train[:15]

Kendi cümlelerimizi deneyelim. Önce tahmini beklediğimiz iki cümle: biri Sabahattin Ali'nin kitaplarına, biri Zweig'ın kitaplarına benziyor.

In [ ]:
sabahattin_ali_gibi = "Berlin'de yalnız yaşayan genç bir adamın, Ankara'da bir mantolu madonna tablosunun ardındaki hikayesi. Köy hayatı, yoksulluk ve aşk."

In [ ]:
sa_vec = vectorizer.transform([sabahattin_ali_gibi])

In [ ]:
model.predict(sa_vec)

In [ ]:
zweig_gibi = "Bir satranç ustasının zihnini ele geçiren saplantı ve Viyana'dan sürgüne giden bir adamın dünyası. Tek bir günde yaşanan tutku."
z_vec = vectorizer.transform([zweig_gibi])

In [ ]:
model.predict(z_vec)

Bir kelimenin hangi yazarın kitaplarında geçtiğine bakmak için:

In [ ]:
df_subset[df_subset.clean.str.contains("satranç")]